# Pure Python Workflow: ASCAT + SMOS-IC + Model

This notebook is a simple Python pipeline for:
1. Building daily paired files (`sm_mod`, `sm_obs`, `idx0`)
2. Computing pentad climatology
3. Computing lagged IVD/IVS skill metrics
4. Computing 3-way TC metrics (ASCAT + SMOS-IC + model)
5. Exporting DA-OL `ΔR` products

All outputs are written as `.npz` (no MATLAB-oriented output formats/naming).

## How To Run

1. Use the `regrid` Python environment as your Jupyter kernel.
2. Run the first setup/config cells.
3. Edit `cfg` paths if needed (`run_roots`, `ascat_root`, `smosic_preprocessed_root`, `output_root`).
4. In the **Runner** cell, run stages in order:

### Pass 1: build daily pairs
```python
RUN_STEP2_ASCAT = True
RUN_STEP2_SMOSIC = True
RUN_STEP3 = False
RUN_STEP4 = False
RUN_TC = False
RUN_STEP5 = False
```

### Pass 2: climatology
```python
RUN_STEP2_ASCAT = False
RUN_STEP2_SMOSIC = False
RUN_STEP3 = True
RUN_STEP4 = False
RUN_TC = False
RUN_STEP5 = False
```

### Pass 3: IVD/IVS
```python
RUN_STEP2_ASCAT = False
RUN_STEP2_SMOSIC = False
RUN_STEP3 = False
RUN_STEP4 = True
RUN_TC = False
RUN_STEP5 = False
```

### Pass 4: TC
```python
RUN_STEP2_ASCAT = False
RUN_STEP2_SMOSIC = False
RUN_STEP3 = False
RUN_STEP4 = False
RUN_TC = True
RUN_STEP5 = False
```

### Pass 5: DA-OL `ΔR`
```python
RUN_STEP2_ASCAT = False
RUN_STEP2_SMOSIC = False
RUN_STEP3 = False
RUN_STEP4 = False
RUN_TC = False
RUN_STEP5 = True
RUN_OL = 'OLv8_M36_cd'
RUN_DA = 'DAv8_M36_cd'
```

## Output Layout

- `output_root/step2_pairs/{ascat|smosic}/{run_name}/YYYYMMDD.npz`
- `output_root/step3_climatology/*.npz`
- `output_root/step4_ivs/*.npz`
- `output_root/step_tc/*.npz`
- `output_root/step5_rdiff/*.npz`



In [ ]:
from __future__ import annotations

import os
import sys
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path

# Optional OpenMP guards (uncomment if your platform needs them)
# os.environ.setdefault('OMP_NUM_THREADS', '1')
# os.environ.setdefault('MKL_NUM_THREADS', '1')
# os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import numpy as np
import scipy.io as sio
from scipy.interpolate import griddata
from netCDF4 import Dataset
import h5py

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'common').exists():
    REPO_ROOT = Path('/discover/nobackup/projects/land_da/geosldas-analysis')

sys.path.insert(0, str(REPO_ROOT / 'common/python/io'))
from read_GEOSldas import read_tilecoord  # type: ignore

print(f'REPO_ROOT={REPO_ROOT}')


In [ ]:
@dataclass
class Config:
    # Date range (inclusive)
    start_date: date = date(2018, 8, 1)
    end_date: date = date(2024, 6, 30)

    domain: str = 'SMAP_EASEv2_M36_GLOBAL'
    out_collection: str = '.tavg24_1d_lnd_Nt.'

    run_roots: dict[str, Path] = None

    ascat_root: Path = Path('/discover/nobackup/qliu/merra_land/DATA/ASCAT_HSAF')
    smosic_preprocessed_root: Path = Path('/discover/nobackup/projects/land_da/SMOS_IC/preprocessed_m36_daily')

    # Root for all Python outputs
    output_root: Path = Path('/discover/nobackup/projects/land_da/Evaluation/IVs/python_output')

    # Processing options
    ascat_interp_method: str = 'linear'
    fill_linear_nans_with_nearest: bool = True
    nlag_days: int = 2
    nmin_ivs: int = 100
    nmin_tc: int = 20

    # Optional TC seasonal subset
    tc_use_all_months: bool = True
    tc_use_summer_only: bool = False  # if tc_use_all_months=False


def default_cfg() -> Config:
    roots = {
        'OLv8_M36_cd': Path('/discover/nobackup/projects/land_da/CYGNSS_Experiments/OLv8_M36_cd'),
        'DAv8_M36_cd': Path('/discover/nobackup/projects/land_da/CYGNSS_Experiments/DAv8_M36_cd'),
    }
    c = Config(run_roots=roots)
    c.output_root.mkdir(parents=True, exist_ok=True)
    return c


cfg = default_cfg()
cfg


In [ ]:
# ---------- Core helpers ----------

def daterange_inclusive(start: date, end: date):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)


def day_tag(d: date) -> str:
    return f'{d.year:04d}{d.month:02d}{d.day:02d}'


def period_tag(start: date, end: date) -> str:
    return f'{start:%Y%m%d}_{end:%Y%m%d}'


def dofyr_nonleap(d: date) -> int:
    if d.month == 2 and d.day == 29:
        d2 = date(2017, 2, 28)
    else:
        d2 = date(2017, d.month, d.day)
    return int(d2.strftime('%j'))


def pentad_1based(d: date) -> int:
    return (dofyr_nonleap(d) - 1) // 5 + 1


def circular_day_distance(a: int, b: int) -> int:
    x = abs(a - b)
    return min(x, 365 - x)


PENTAD_CENTERS_DOY = np.arange(3, 366, 5, dtype=np.int32)


def build_doy_to_pentads(window_days: int = 15):
    out = {}
    for doy in range(1, 366):
        keep = [i for i, c in enumerate(PENTAD_CENTERS_DOY) if circular_day_distance(doy, int(c)) <= window_days]
        out[doy] = np.array(keep, dtype=np.int32)
    return out


DOY_TO_PENTADS = build_doy_to_pentads(15)


def ease2_m36_lon_lat() -> tuple[np.ndarray, np.ndarray]:
    # Pure-python EASE2 M36 inverse for full 964x406 grid
    nlon, nlat = 964, 406
    r0 = (nlon - 1) / 2.0
    s0 = (nlat - 1) / 2.0
    map_scale_m = 36032.220840584

    row2d = np.repeat(np.arange(nlat, dtype=np.float64)[None, :], nlon, axis=0)
    col2d = np.repeat(np.arange(nlon, dtype=np.float64)[:, None], nlat, axis=1)

    x = (col2d - r0) * map_scale_m
    y = (s0 - row2d) * map_scale_m

    a = 6378137.0
    e = 0.081819190843
    e2 = e ** 2

    phi1 = np.deg2rad(30.0)
    kz = np.cos(phi1) / np.sqrt(1.0 - e2 * np.sin(phi1) ** 2)

    qp = (1.0 - e2) * ((1.0 / (1.0 - e2)) - (1.0 / (2.0 * e)) * np.log((1.0 - e) / (1.0 + e)))

    beta = np.arcsin(2.0 * y * kz / (a * qp))
    lam = x / (a * kz)

    e4 = e ** 4
    e6 = e ** 6

    phi = (
        beta
        + (((e2 / 3.0) + ((31.0 / 180.0) * e4) + ((517.0 / 5040.0) * e6)) * np.sin(2.0 * beta))
        + ((((23.0 / 360.0) * e4) + ((251.0 / 3780.0) * e6)) * np.sin(4.0 * beta))
        + (((761.0 / 45360.0) * e6) * np.sin(6.0 * beta))
    )

    lat = np.rad2deg(phi)
    lon = np.rad2deg(lam)
    lon = np.where(lon < -180.0, lon + 360.0, lon)
    lon = np.where(lon > 180.0, lon - 360.0, lon)

    return lon.astype(np.float64), lat.astype(np.float64)


def tilecoord_path_for_run(run_root: Path, run_name: str, domain: str) -> Path:
    cands = [
        run_root / 'output' / domain / 'rc_out' / f'{run_name}.ldas_tilecoord.bin',
        run_root / run_name / 'output' / domain / 'rc_out' / f'{run_name}.ldas_tilecoord.bin',
        run_root / f'{run_name}.ldas_tilecoord.bin',
    ]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError('Could not find tilecoord. Checked: ' + ', '.join(str(p) for p in cands))


def read_model_tilecoord(run_root: Path, run_name: str, domain: str):
    p = tilecoord_path_for_run(run_root, run_name, domain)
    tc = read_tilecoord(str(p))
    i = np.asarray(tc['i_indg'], dtype=np.int64)
    j = np.asarray(tc['j_indg'], dtype=np.int64)
    n_tile = int(tc['N_tile'])
    return {'path': p, 'i_indg': i, 'j_indg': j, 'N_tile': n_tile}


def load_mat(path: Path, fields: tuple[str, ...]):
    # For ASCAT AD input mats
    try:
        d = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
        return {f: np.asarray(d[f]) for f in fields}
    except NotImplementedError:
        out = {}
        with h5py.File(path, 'r') as h5:
            for f in fields:
                out[f] = np.asarray(h5[f]).squeeze()
        return out


def save_pair_npz(path: Path, sm_mod_2d: np.ndarray, sm_obs_2d: np.ndarray):
    sm_obs = sm_obs_2d.copy()
    sm_mod = sm_mod_2d.copy()
    sm_obs[~np.isfinite(sm_mod)] = np.nan
    sm_mod[~np.isfinite(sm_obs)] = np.nan

    flat_obs = sm_obs.ravel(order='F')
    flat_mod = sm_mod.ravel(order='F')
    idx0 = np.flatnonzero(np.isfinite(flat_obs)).astype(np.int32)

    np.savez_compressed(
        path,
        idx0=idx0,
        sm_obs=flat_obs[idx0].astype(np.float32),
        sm_mod=flat_mod[idx0].astype(np.float32),
    )


def load_pair_npz(path: Path):
    z = np.load(path)
    return z['idx0'].astype(np.int64), z['sm_mod'].astype(np.float64), z['sm_obs'].astype(np.float64)


def pair_npz_path(prefix: str, run_name: str, d: date, cfg: Config) -> Path:
    p = cfg.output_root / 'step2_pairs' / prefix / run_name
    p.mkdir(parents=True, exist_ok=True)
    return p / f'{day_tag(d)}.npz'


lon_m36, lat_m36 = ease2_m36_lon_lat()
Nlon, Nlat = lon_m36.shape
Ncell = Nlon * Nlat
print(f'M36 grid: {Nlon} x {Nlat} ({Ncell} cells)')


In [ ]:
# ---------- Step 2: build daily pair npz files ----------

def model_daily_candidates(run_root: Path, run_name: str, domain: str, out_collection: str, d: date):
    y, m, day = f'{d.year:04d}', f'{d.month:02d}', f'{d.day:02d}'
    return [
        run_root / 'output' / domain / 'cat' / 'ens0000' / f'Y{y}' / f'M{m}' / f'{run_name}{out_collection}{y}{m}{day}_1200z.nc4',
        run_root / 'output' / domain / 'cat' / 'ens_avg'  / f'Y{y}' / f'M{m}' / f'{run_name}{out_collection}{y}{m}{day}_1200z.nc4',
        run_root / 'output' / domain / 'cat' / 'ens0000' / f'Y{y}' / f'M{m}' / f'{run_name}{out_collection}{y}{m}{day}.nc4',
        run_root / 'output' / domain / 'cat' / 'ens_avg'  / f'Y{y}' / f'M{m}' / f'{run_name}{out_collection}{y}{m}{day}.nc4',
    ]


def read_nc_var(path: Path, var: str) -> np.ndarray:
    with Dataset(path, 'r') as ds:
        if var not in ds.variables:
            raise KeyError(f'{var} not found in {path}')
        x = np.asarray(ds.variables[var][:], dtype=np.float64)
    x[x < 0] = np.nan
    return x


def orient_sfmc(arr: np.ndarray, n_tile: int) -> np.ndarray:
    arr = np.squeeze(np.asarray(arr, dtype=np.float64))
    if arr.ndim == 1:
        if arr.shape[0] != n_tile:
            raise RuntimeError(f'SFMC len {arr.shape[0]} != n_tile {n_tile}')
        return arr[:, None]
    if arr.ndim == 2:
        if arr.shape[0] == n_tile:
            return arr
        if arr.shape[1] == n_tile:
            return arr.T
    raise RuntimeError(f'Unexpected SFMC shape {arr.shape}; n_tile={n_tile}')


def read_model_daily_mean(d: date, run_root: Path, run_name: str, cfg: Config, tc: dict) -> np.ndarray:
    for f in model_daily_candidates(run_root, run_name, cfg.domain, cfg.out_collection, d):
        if f.exists():
            sfmc = read_nc_var(f, 'SFMC')
            sf = orient_sfmc(sfmc, tc['N_tile'])
            return np.nanmean(sf, axis=1)

    # fallback hourly
    hr_list = [1, 4, 7, 10, 13, 16, 19, 22]
    y, m, day = f'{d.year:04d}', f'{d.month:02d}', f'{d.day:02d}'
    rows = []

    for hh in hr_list:
        hhmm = f'{hh:02d}30'
        name = f'{run_name}{cfg.out_collection}{y}{m}{day}_{hhmm}z.nc4'
        p1 = run_root / 'output' / cfg.domain / 'cat' / 'ens0000' / f'Y{y}' / f'M{m}' / name
        p2 = run_root / 'output' / cfg.domain / 'cat' / 'ens_avg' / f'Y{y}' / f'M{m}' / name
        f = p1 if p1.exists() else p2
        if not f.exists():
            raise FileNotFoundError(f'Missing hourly model file: {p1} or {p2}')
        x = read_nc_var(f, 'sm_surface').reshape(-1)
        if x.shape[0] != tc['N_tile']:
            raise RuntimeError(f'sm_surface len {x.shape[0]} != n_tile {tc["N_tile"]} in {f}')
        rows.append(x)

    return np.nanmean(np.stack(rows, axis=1), axis=1)


def map_tile_to_m36(sm_tile: np.ndarray, tc: dict) -> np.ndarray:
    out = np.full((Nlon, Nlat), np.nan, dtype=np.float64)
    out[tc['i_indg'], tc['j_indg']] = sm_tile
    return out


def read_ascat_land_info(ascat_root: Path):
    f = ascat_root / 'Auxiliary' / 'TUW_WARP5_grid_info_2_2.nc'
    with Dataset(f, 'r') as ds:
        land_flag = np.asarray(ds.variables['land_flag'][:]).reshape(-1)
        lon = np.asarray(ds.variables['lon'][:], dtype=np.float64).reshape(-1)
        lat = np.asarray(ds.variables['lat'][:], dtype=np.float64).reshape(-1)
    m = land_flag == 1
    return lon[m], lat[m]


def read_ascat_obs_m36(d: date, cfg: Config, lon_gpi_land: np.ndarray, lat_gpi_land: np.ndarray) -> np.ndarray:
    f = cfg.ascat_root / 'H119_H120_processed' / f'Y{d.year:04d}' / f'M{d.month:02d}' / f'ASCAT_HSAF_H119_SM_{day_tag(d)}_AD.mat'
    out = np.full((Nlon, Nlat), np.nan, dtype=np.float64)
    if not f.exists():
        return out

    dct = load_mat(f, ('sm_tile', 'conf_flag_tile'))
    sm = np.asarray(dct['sm_tile'], dtype=np.float64).reshape(-1)
    conf = np.asarray(dct['conf_flag_tile']).reshape(-1)

    if sm.shape[0] != lon_gpi_land.shape[0]:
        raise RuntimeError(f'ASCAT sm_tile len {sm.shape[0]} != land-grid len {lon_gpi_land.shape[0]} ({f})')

    sm[sm > 100] = np.nan
    sm[conf >= 1] = np.nan

    v = np.isfinite(sm)
    if np.count_nonzero(v) < 3:
        return out

    points = np.column_stack((lon_gpi_land[v], lat_gpi_land[v]))
    values = sm[v]

    method = cfg.ascat_interp_method.lower()
    out = griddata(points, values, (lon_m36, lat_m36), method=method)

    if method == 'linear' and cfg.fill_linear_nans_with_nearest:
        fill = griddata(points, values, (lon_m36, lat_m36), method='nearest')
        out = np.where(np.isfinite(out), out, fill)

    return out


def read_smosic_obs_m36(d: date, cfg: Config) -> np.ndarray:
    f = cfg.smosic_preprocessed_root / f'smos_ic_sm_m36_{day_tag(d)}.nc'
    out = np.full((Nlon, Nlat), np.nan, dtype=np.float64)
    if not f.exists():
        return out

    with Dataset(f, 'r') as ds:
        idx0 = np.asarray(ds.variables['idx_EASEv2_lonxlat'][:], dtype=np.int64).reshape(-1)
        vals = np.asarray(ds.variables['sm_obs'][:], dtype=np.float64).reshape(-1)

    good = (idx0 >= 0) & (idx0 < Ncell) & np.isfinite(vals)
    if np.any(good):
        flat = out.ravel(order='F')
        flat[idx0[good]] = vals[good]
        out = flat.reshape((Nlon, Nlat), order='F')

    return out


def run_step2(sensor: str, run_name: str, run_root: Path, cfg: Config):
    assert sensor in ('ascat', 'smosic')

    tc = read_model_tilecoord(run_root, run_name, cfg.domain)

    lon_gpi_land = lat_gpi_land = None
    if sensor == 'ascat':
        lon_gpi_land, lat_gpi_land = read_ascat_land_info(cfg.ascat_root)

    n_days = 0
    n_obs_all_nan = 0

    for d in daterange_inclusive(cfg.start_date, cfg.end_date):
        sm_mod_tile = read_model_daily_mean(d, run_root, run_name, cfg, tc)
        sm_mod_2d = map_tile_to_m36(sm_mod_tile, tc)

        if sensor == 'ascat':
            sm_obs_2d = read_ascat_obs_m36(d, cfg, lon_gpi_land, lat_gpi_land)
        else:
            sm_obs_2d = read_smosic_obs_m36(d, cfg)

        if not np.isfinite(sm_obs_2d).any():
            n_obs_all_nan += 1

        save_pair_npz(pair_npz_path(sensor, run_name, d, cfg), sm_mod_2d, sm_obs_2d)
        n_days += 1

        if n_days % 100 == 0:
            print(f'[step2 {sensor} {run_name}] {n_days} days written...')

    print(f'[step2 {sensor} {run_name}] done: days={n_days}, days_obs_all_nan={n_obs_all_nan}')


In [ ]:
# ---------- Step 3: pentad climatology ----------

def climatology_npz_path(sensor: str, run_name: str, cfg: Config) -> Path:
    p = cfg.output_root / 'step3_climatology'
    p.mkdir(parents=True, exist_ok=True)
    return p / f'{sensor}_climatology_{run_name}_{period_tag(cfg.start_date, cfg.end_date)}.npz'


def run_step3(sensor: str, run_name: str, cfg: Config):
    npen = len(PENTAD_CENTERS_DOY)

    mod_sum = np.zeros((Ncell, npen), dtype=np.float32)
    obs_sum = np.zeros((Ncell, npen), dtype=np.float32)
    n_sum = np.zeros((Ncell, npen), dtype=np.int32)

    nday_min = 4 * (cfg.end_date.year - cfg.start_date.year)

    for i, d in enumerate(daterange_inclusive(cfg.start_date, cfg.end_date), start=1):
        f = pair_npz_path(sensor, run_name, d, cfg)
        if not f.exists():
            continue

        idx0, sm_mod, sm_obs = load_pair_npz(f)
        if idx0.size == 0:
            continue

        pidx = DOY_TO_PENTADS[dofyr_nonleap(d)]
        for p in pidx:
            mod_sum[idx0, p] += sm_mod.astype(np.float32)
            obs_sum[idx0, p] += sm_obs.astype(np.float32)
            n_sum[idx0, p] += 1

        if i % 200 == 0:
            print(f'[step3 {sensor} {run_name}] day {i}')

    c = n_sum.astype(np.float32)
    valid = c >= float(nday_min)

    mod_clim = np.full((Ncell, npen), np.nan, dtype=np.float32)
    obs_clim = np.full((Ncell, npen), np.nan, dtype=np.float32)
    mod_clim[valid] = mod_sum[valid] / c[valid]
    obs_clim[valid] = obs_sum[valid] / c[valid]

    out = climatology_npz_path(sensor, run_name, cfg)
    np.savez_compressed(out, mod_clim=mod_clim, obs_clim=obs_clim, n_clim=n_sum, nday_min=np.int32(nday_min))
    print(f'Wrote: {out}')
    return out


In [ ]:
# ---------- Step 4: IVD/IVS ----------

def ivs_npz_path(sensor: str, run_name: str, cfg: Config) -> Path:
    p = cfg.output_root / 'step4_ivs'
    p.mkdir(parents=True, exist_ok=True)
    return p / f'{sensor}_ivs_lag{cfg.nlag_days}_{run_name}_{period_tag(cfg.start_date, cfg.end_date)}.npz'


def run_step4(sensor: str, run_name: str, cfg: Config):
    cfile = climatology_npz_path(sensor, run_name, cfg)
    z = np.load(cfile)
    mod_clim = z['mod_clim'].astype(np.float64)
    obs_clim = z['obs_clim'].astype(np.float64)

    nlag = int(cfg.nlag_days)
    nmin = int(cfg.nmin_ivs)

    mod_sm_sum = np.zeros(Ncell, dtype=np.float64)
    mod_sm2_sum = np.zeros(Ncell, dtype=np.float64)
    obs_sm_sum = np.zeros(Ncell, dtype=np.float64)
    obs_sm2_sum = np.zeros(Ncell, dtype=np.float64)

    modxobs_sm_sum = np.zeros(Ncell, dtype=np.float64)
    modxlag_sm_sum = np.zeros(Ncell, dtype=np.float64)
    obsxlag_sm_sum = np.zeros(Ncell, dtype=np.float64)
    obslagxmod_sm_sum = np.zeros(Ncell, dtype=np.float64)
    modlagxobs_sm_sum = np.zeros(Ncell, dtype=np.float64)
    n_sm = np.zeros(Ncell, dtype=np.int32)

    d0 = cfg.start_date + timedelta(days=nlag)
    for i, d in enumerate(daterange_inclusive(d0, cfg.end_date), start=1):
        dp = d - timedelta(days=nlag)

        f_now = pair_npz_path(sensor, run_name, d, cfg)
        f_pre = pair_npz_path(sensor, run_name, dp, cfg)
        if not (f_now.exists() and f_pre.exists()):
            continue

        idx_n, mod_n, obs_n = load_pair_npz(f_now)
        idx_p, mod_p, obs_p = load_pair_npz(f_pre)
        if idx_n.size == 0 or idx_p.size == 0:
            continue

        idx, inow, ipre = np.intersect1d(idx_n, idx_p, assume_unique=False, return_indices=True)
        if idx.size == 0:
            continue

        p_now = pentad_1based(d) - 1
        p_pre = pentad_1based(dp) - 1

        sm_mod = mod_n[inow] - mod_clim[idx, p_now]
        sm_obs = obs_n[inow] - obs_clim[idx, p_now]
        sm_mod_pre = mod_p[ipre] - mod_clim[idx, p_pre]
        sm_obs_pre = obs_p[ipre] - obs_clim[idx, p_pre]

        iv = np.isfinite(sm_mod)  # match MATLAB logic
        if not np.any(iv):
            continue

        ii = idx[iv]
        m = sm_mod[iv]
        o = sm_obs[iv]
        mp = sm_mod_pre[iv]
        op = sm_obs_pre[iv]

        mod_sm_sum[ii] += m
        mod_sm2_sum[ii] += m * m
        obs_sm_sum[ii] += o
        obs_sm2_sum[ii] += o * o

        modxobs_sm_sum[ii] += m * o
        modxlag_sm_sum[ii] += m * mp
        obsxlag_sm_sum[ii] += o * op
        obslagxmod_sm_sum[ii] += m * op
        modlagxobs_sm_sum[ii] += mp * o

        n_sm[ii] += 1

        if i % 200 == 0:
            print(f'[step4 {sensor} {run_name}] day {i}')

    NN = n_sm.astype(np.float64)
    NN[NN < nmin] = np.nan

    mod_mean = mod_sm_sum / NN
    obs_mean = obs_sm_sum / NN

    C_mod_mod = mod_sm2_sum / NN - mod_mean ** 2
    C_obs_obs = obs_sm2_sum / NN - obs_mean ** 2
    C_mod_mod[C_mod_mod < 0] = np.nan
    C_obs_obs[C_obs_obs < 0] = np.nan

    C_mod_obs = modxobs_sm_sum / NN - mod_mean * obs_mean

    R_mod_obs = C_mod_obs / np.sqrt(C_mod_mod * C_obs_obs)
    R_mod_obs[~np.isfinite(R_mod_obs)] = -9999.0

    C_mod_obs[C_mod_obs < 0] = np.nan

    C_mod_modlag = modxlag_sm_sum / NN - mod_mean ** 2
    C_obs_obslag = obsxlag_sm_sum / NN - obs_mean ** 2
    C_mod_modlag[C_mod_modlag < 0] = np.nan
    C_obs_obslag[C_obs_obslag < 0] = np.nan

    C_mod_obslag = obslagxmod_sm_sum / NN - mod_mean * obs_mean
    C_modlag_obs = modlagxobs_sm_sum / NN - mod_mean * obs_mean

    S_ivd = np.sqrt(C_mod_modlag / C_obs_obslag)
    S_ivs_obs = C_mod_obslag / C_obs_obslag
    S_ivs_mod = C_mod_modlag / C_modlag_obs

    R2_ivd_mod = C_mod_obs * S_ivd / C_mod_mod
    R2_ivd_obs = C_mod_obs / C_obs_obs / S_ivd
    R2_ivs_mod = C_mod_obs * S_ivs_obs / C_mod_mod
    R2_ivs_obs = C_mod_obs / C_obs_obs / S_ivs_obs

    for arr in (R2_ivd_mod, R2_ivd_obs, R2_ivs_mod, R2_ivs_obs):
        arr[arr < 0] = np.nan
        arr[arr > 1] = 1.0

    out = ivs_npz_path(sensor, run_name, cfg)
    np.savez_compressed(
        out,
        n_sm=n_sm,
        nmin=np.int32(nmin),
        nlag=np.int32(nlag),
        R2_ivd_mod=R2_ivd_mod,
        R2_ivd_obs=R2_ivd_obs,
        R2_ivs_mod=R2_ivs_mod,
        R2_ivs_obs=R2_ivs_obs,
        R_mod_obs=R_mod_obs,
    )
    print(f'Wrote: {out}')
    return out


In [ ]:
# ---------- TC (ASCAT + SMOS-IC + model) + Step 5 ΔR ----------

def tc_npz_path(run_name: str, cfg: Config) -> Path:
    p = cfg.output_root / 'step_tc'
    p.mkdir(parents=True, exist_ok=True)
    season = 'all' if cfg.tc_use_all_months else ('summer' if cfg.tc_use_summer_only else 'winter')
    return p / f'tc_ascat_smosic_{run_name}_{period_tag(cfg.start_date, cfg.end_date)}_{season}.npz'


def run_tc(run_name: str, cfg: Config):
    zA = np.load(climatology_npz_path('ascat', run_name, cfg))
    zS = np.load(climatology_npz_path('smosic', run_name, cfg))

    mod_clim = zA['mod_clim'].astype(np.float64)
    asc_clim = zA['obs_clim'].astype(np.float64) / 200.0
    smos_clim = zS['obs_clim'].astype(np.float64)

    nmin = int(cfg.nmin_tc)

    SMOS_sm_sum = np.zeros(Ncell, dtype=np.float64)
    SMOS_sm2_sum = np.zeros(Ncell, dtype=np.float64)
    mod_sm_sum = np.zeros(Ncell, dtype=np.float64)
    mod_sm2_sum = np.zeros(Ncell, dtype=np.float64)
    ASC_sm_sum = np.zeros(Ncell, dtype=np.float64)
    ASC_sm2_sum = np.zeros(Ncell, dtype=np.float64)

    modxASC = np.zeros(Ncell, dtype=np.float64)
    SMOSxASC = np.zeros(Ncell, dtype=np.float64)
    SMOSxmod = np.zeros(Ncell, dtype=np.float64)
    N_sm = np.zeros(Ncell, dtype=np.int32)

    for i, d in enumerate(daterange_inclusive(cfg.start_date, cfg.end_date), start=1):
        fA = pair_npz_path('ascat', run_name, d, cfg)
        fS = pair_npz_path('smosic', run_name, d, cfg)
        if not (fA.exists() and fS.exists()):
            continue

        idxA, modA, obsA = load_pair_npz(fA)
        idxS, modS, obsS = load_pair_npz(fS)
        if idxA.size == 0 or idxS.size == 0:
            continue

        idx, ia, ib = np.intersect1d(idxS, idxA, assume_unique=False, return_indices=True)
        if idx.size == 0:
            continue

        sm_mod = modS[ia]
        sm_smos = obsS[ia]
        sm_asc = obsA[ib] / 200.0

        if not cfg.tc_use_all_months:
            if cfg.tc_use_summer_only:
                mkeep = (d.month >= 5 and d.month <= 9)
            else:
                mkeep = (d.month <= 4 or d.month >= 10)
            if not mkeep:
                continue

        p = pentad_1based(d) - 1

        sm_smos = sm_smos - smos_clim[idx, p]
        sm_mod = sm_mod - mod_clim[idx, p]
        sm_asc = sm_asc - asc_clim[idx, p]

        iv = np.isfinite(sm_mod) & np.isfinite(sm_smos) & np.isfinite(sm_asc)
        if not np.any(iv):
            continue

        ii = idx[iv]
        m = sm_mod[iv]
        s = sm_smos[iv]
        a = sm_asc[iv]

        mod_sm_sum[ii] += m
        mod_sm2_sum[ii] += m * m
        SMOS_sm_sum[ii] += s
        SMOS_sm2_sum[ii] += s * s
        ASC_sm_sum[ii] += a
        ASC_sm2_sum[ii] += a * a

        SMOSxASC[ii] += s * a
        SMOSxmod[ii] += s * m
        modxASC[ii] += m * a
        N_sm[ii] += 1

        if i % 200 == 0:
            print(f'[TC {run_name}] day {i}')

    NN = N_sm.astype(np.float64)
    NN[NN < nmin] = np.nan

    SMOS_mean = SMOS_sm_sum / NN
    mod_mean = mod_sm_sum / NN
    ASC_mean = ASC_sm_sum / NN

    C_SMOS_SMOS = SMOS_sm2_sum / NN - SMOS_mean ** 2
    C_mod_mod = mod_sm2_sum / NN - mod_mean ** 2
    C_ASC_ASC = ASC_sm2_sum / NN - ASC_mean ** 2

    C_SMOS_SMOS[C_SMOS_SMOS < 0] = np.nan
    C_mod_mod[C_mod_mod < 0] = np.nan
    C_ASC_ASC[C_ASC_ASC < 0] = np.nan

    C_mod_ASC = modxASC / NN - mod_mean * ASC_mean
    C_SMOS_mod = SMOSxmod / NN - SMOS_mean * mod_mean
    C_SMOS_ASC = SMOSxASC / NN - SMOS_mean * ASC_mean

    R_mod_smos = C_SMOS_mod / np.sqrt(C_mod_mod * C_SMOS_SMOS)
    R_mod_asc = C_mod_ASC / np.sqrt(C_mod_mod * C_ASC_ASC)
    R_asc_smos = C_SMOS_ASC / np.sqrt(C_SMOS_SMOS * C_ASC_ASC)

    R2_tc_smos = C_SMOS_mod * C_SMOS_ASC / C_mod_ASC / C_SMOS_SMOS
    R2_tc_mod = C_SMOS_mod * C_mod_ASC / C_SMOS_ASC / C_mod_mod
    R2_tc_asc = C_mod_ASC * C_SMOS_ASC / C_SMOS_mod / C_ASC_ASC

    for arr in (R2_tc_smos, R2_tc_mod, R2_tc_asc):
        arr[arr < 0.001] = np.nan
        arr[arr > 1.0] = 1.0

    sigma2_smos = C_SMOS_SMOS - C_SMOS_ASC * C_SMOS_mod / C_mod_ASC
    sigma2_mod = C_mod_mod - C_mod_ASC * C_SMOS_mod / C_SMOS_ASC
    sigma2_asc = C_ASC_ASC - C_SMOS_ASC * C_mod_ASC / C_SMOS_mod

    out = tc_npz_path(run_name, cfg)
    np.savez_compressed(
        out,
        lon=lon_m36.astype(np.float32),
        lat=lat_m36.astype(np.float32),
        n_sm=N_sm,
        nmin=np.int32(nmin),
        R2_tc_smos=R2_tc_smos,
        R2_tc_asc=R2_tc_asc,
        R2_tc_mod=R2_tc_mod,
        sigma2_smos=sigma2_smos,
        sigma2_mod=sigma2_mod,
        sigma2_asc=sigma2_asc,
        R_mod_smos=R_mod_smos,
        R_mod_asc=R_mod_asc,
        R_asc_smos=R_asc_smos,
    )
    print(f'Wrote: {out}')
    return out


def rdiff_npz_path(sensor: str, d1_run: str, d2_run: str, cfg: Config) -> Path:
    p = cfg.output_root / 'step5_rdiff'
    p.mkdir(parents=True, exist_ok=True)
    return p / f'rdiff_{sensor}_{d2_run}_minus_{d1_run}_{period_tag(cfg.start_date, cfg.end_date)}.npz'


def run_step5_rdiff(sensor: str, d1_run: str, d2_run: str, cfg: Config):
    z1 = np.load(ivs_npz_path(sensor, d1_run, cfg))
    z2 = np.load(ivs_npz_path(sensor, d2_run, cfg))

    R1 = np.sqrt(np.asarray(z1['R2_ivs_mod'], dtype=np.float64).reshape(-1))
    R2 = np.sqrt(np.asarray(z2['R2_ivs_mod'], dtype=np.float64).reshape(-1))
    R_obs = np.sqrt(np.asarray(z2['R2_ivs_obs'], dtype=np.float64).reshape(-1))

    R1[R1 < 0.1] = np.nan
    R2[R2 < 0.1] = np.nan
    R_obs[R_obs < 0.1] = np.nan

    bad = ~np.isfinite(R_obs)
    R1[bad] = np.nan
    R2[bad] = np.nan

    dr = (R2 - R1).astype(np.float32)

    out = rdiff_npz_path(sensor, d1_run, d2_run, cfg)
    np.savez_compressed(out, delta_r=dr, lon=lon_m36.astype(np.float32), lat=lat_m36.astype(np.float32))
    print(f'Wrote: {out} | mean ΔR={np.nanmean(dr):.6f}')
    return out


In [ ]:
# ---------- Runner ----------
# Set switches and run this cell.

RUN_STEP2_ASCAT = False
RUN_STEP2_SMOSIC = False
RUN_STEP3 = False
RUN_STEP4 = False
RUN_TC = False
RUN_STEP5 = False

RUN_OL = 'OLv8_M36_cd'
RUN_DA = 'DAv8_M36_cd'

if RUN_STEP2_ASCAT:
    for run_name, run_root in cfg.run_roots.items():
        run_step2('ascat', run_name, run_root, cfg)

if RUN_STEP2_SMOSIC:
    for run_name, run_root in cfg.run_roots.items():
        run_step2('smosic', run_name, run_root, cfg)

if RUN_STEP3:
    for run_name in cfg.run_roots:
        run_step3('ascat', run_name, cfg)
        run_step3('smosic', run_name, cfg)

if RUN_STEP4:
    for run_name in cfg.run_roots:
        run_step4('ascat', run_name, cfg)
        run_step4('smosic', run_name, cfg)

if RUN_TC:
    for run_name in cfg.run_roots:
        run_tc(run_name, cfg)

if RUN_STEP5:
    run_step5_rdiff('ascat', RUN_OL, RUN_DA, cfg)
    run_step5_rdiff('smosic', RUN_OL, RUN_DA, cfg)

print('Done')
